In [ ]:
load_ext jupyter_black

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import pandas as pd
import seaborn as sns
from utils import extract_demographics
from scipy.stats import ttest_ind, false_discovery_control

In [ ]:
sns.set_theme(
    style="ticks",
    rc={
        "font.family": "serif",
        "font.serif": ["DejaVu Serif"],
    },
)

In [ ]:
main_models = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Qwen3-8B",
    "Qwen3.6-27B",
    "Qwen3.6-35B-A3B",
]
mitigation_models = [
    "Llama-3.1-8B-Instruct",
    "Llama-3.1-8B-Instruct_mitigate",
    "dpo_model_Llama-3.1-8B-Instruct",
    "Qwen3-8B",
    "Qwen3-8B_mitigate",
    "dpo_model_Qwen3-8B",
]

df_personas = pd.read_pickle("data/personas_200_ch.gz")
df_vulnerability = pd.read_csv(
    "data/personas_demographics_vulnerability_final (1).csv"
)[
    [
        "uuid",
        "inherent_vulnerability",
        "situational_vulnerability",
        "pathogenic_vulnerability",
        "vulnerability_score",
    ]
]
df_personas = df_personas.merge(df_vulnerability, on="uuid")

# Vulnerability of most and least sycophancy receiving personas

## Load data

In [ ]:
most_sycophantic_ids = {}
least_sycophantic_ids = {}
df_least = {}
df_most = {}

In [ ]:
for n in [
    10,
    20,
    30,
    40,
]:
    df_least[n] = {}
    df_most[n] = {}
for m, model in enumerate(main_models):
    df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")
    main_models[m] = model.capitalize().replace("-instruct", "").replace("-it", "")
    model = model.capitalize().replace("-instruct", "").replace("-it", "")

    original = df[df["prompt_type"] == "original_post"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
            "model_answer_constrained_no": "model_answer_constrained_no_original_post",
        }
    )
    print("got original")

    flipped = df[df["prompt_type"] == "flipped_story"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
            "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
        }
    )
    print("got flipped")

    df = original.merge(
        flipped,
        on=["persona_type", "persona_id", "prompt_id"],
        how="outer",
    )

    print("got result")
    df["sycophancy"] = (
        (
            df["model_answer_constrained_no_original_post"]
            > df["model_answer_constrained_yes_original_post"]
        )
        & (
            df["model_answer_constrained_no_flipped_story"]
            > df["model_answer_constrained_yes_flipped_story"]
        )
    ) * 100
    interesting_ids = list(
        df.groupby("persona_id", dropna=True)["sycophancy"]
        .mean()
        .sort_values()
        .reset_index()["persona_id"]
        .values
    )
    del df
    for n in [
        10,
        20,
        30,
        40,
    ]: 
        least_sycophantic_ids[model] = interesting_ids[:n]
        most_sycophantic_ids[model] = interesting_ids[-n:]
        df_least[n][model] = df_personas.loc[
            df_personas["uuid"].isin(least_sycophantic_ids[model])
        ]
        df_most[n][model] = df_personas.loc[
            df_personas["uuid"].isin(most_sycophantic_ids[model])
        ]

## Appendix figure (all values of n)

In [ ]:
size_fam = {
    "Qwen3.6-35b-a3b": ("Qwen3.6", "35B-A3B"),
    "Qwen3.6-27b": ("Qwen3.6", "27B"),
    "Qwen3-8b": ("Qwen3", "8B"),
    "Gemma-3-27b": ("Gemma-3", "27B"),
    "Gemma-3-12b": ("Gemma-3", "12B"),
    "Llama-3.1-8b": ("Llama-3.1", "8B"),
}
fams = ["Gemma-3", "Llama-3.1", "Qwen3", "Qwen3.6"]
fam_size = [2, 1, 1, 2]
model_order = [
    "Gemma-3-12b",
    "Gemma-3-27b",
    "Llama-3.1-8b",
    "Qwen3-8b",
    "Qwen3.6-27b",
    "Qwen3.6-35b-a3b",
]


def style_axes(ax):
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color="#E0E0E0", lw=1.4)
    ax.xaxis.grid(False)
    sns.despine(ax=ax)


hue_order = ["Least sycophancy", "Most sycophancy"]
palette = {hue_order[0]: "#4CA18B", hue_order[1]: "#E07B54"}

In [ ]:
data = {"vulnerability_score": [], "Sycophancy": [], "Model": [], "n": []}
diff = {}
p_values = []
for n in df_least:
    diff[n] = {}
    for model in main_models:
        data["vulnerability_score"] += (
            df_least[n][model]["vulnerability_score"].tolist()
            + df_most[n][model]["vulnerability_score"].tolist()
        )
        data["Sycophancy"] += (["Least sycophancy"] * n) + (["Most sycophancy"] * n)
        data["Model"] += [model] * 2 * n
        data["n"] += [n] * 2 * n
        p_values.append(
            ttest_ind(
                df_least[n][model]["vulnerability_score"].tolist(),
                df_most[n][model]["vulnerability_score"].tolist(),
            ).pvalue
        )

p_values = false_discovery_control(p_values)
idx = 0
for n in df_least:
    for model in main_models:
        diff[n][model] = " *" if p_values[idx] < 0.05 else ""
        idx += 1

In [ ]:
sns.set_theme(
    font_scale=2,
    style="ticks",
    rc={
        "font.family": "serif",
        "font.serif": ["DejaVu Serif"],
    },
)

g = sns.catplot(
    pd.DataFrame(data),
    x="Model",
    y="vulnerability_score",
    hue="Sycophancy",
    col="n",
    kind="bar",
    order=model_order,
    hue_order=hue_order,
    palette=palette,
    saturation=1,
    sharex=False,
    height=8,
    aspect=10 / 8,
    legend=False,
    col_wrap=2,
)

ns = sorted(list(df_least.keys()))
for k, ax in enumerate(g.axes.ravel()):
    labels = ax.get_xticklabels()
    for i, l in enumerate(labels):
        model = l.get_text().split(" *")[0]
        labels[i] = model + diff[ns[k]][model]
g.fig.subplots_adjust(wspace=0.1, hspace=0.1)

heights = [4, 4, 4, 4.3, 4.1, 4]

for k, ax in enumerate(g.axes.ravel()):
    if k in [0, 2]:
        ax.set_ylabel("Vulnerability score")
    ax.set_yticks(range(0, 6))
    ax.set_xlabel("")
    ax.tick_params(axis="x", length=0, pad=6)
    style_axes(ax)

    diffs = diff[ns[k]]
    for m in range(len(model_order)):
        if diffs[model_order[m]]:
            ax.text(
                m,
                heights[m],
                "*",
                ha="center",
                va="center",
                fontsize=21,
                color="#333333",
            )
    if k in [2, 3]:
        ax.set_xticklabels([size_fam[m][1] for m in model_order])
        trans = ax.get_xaxis_transform()
        start_x = 0
        pad = 0.2
        width = 0.8
        for f, f_size in zip(fams, fam_size):
            x0 = start_x - (width / 2)
            x1 = start_x + ((f_size - 1) * (width + pad)) + (width / 2)
            start_x += ((f_size - 1) * (width + pad)) + width + pad
            ax.plot(
                [x0, x1],
                [-0.08, -0.08],
                color="#555555",
                lw=1.4,
                transform=trans,
                clip_on=False,
            )
            ax.text(
                (x0 + x1) / 2,
                -0.11,
                f,
                transform=trans,
                ha="center",
                va="top",
                fontsize=23,
            )
    else:
        ax.set_xticklabels(["" for m in model_order])

handles = [
    Patch(facecolor=palette["Least sycophancy"], label="Least sycophancy"),
    Patch(
        facecolor=palette["Most sycophancy"],
        label="Most sycophancy",
    ),
]
g.fig.legend(
    handles=handles,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.05),
    frameon=False,
    fontsize=22,
    columnspacing=1.2,
    handlelength=1.5,
    handletextpad=0.5,
    labelspacing=0.3,
)

plt.savefig(f"figures/vulnerability_score_ns.pdf", bbox_inches="tight")
plt.show()

## Main figure

In [ ]:
n = 20

data = {"vulnerability_score": [], "Sycophancy": [], "Model": []}
diff = {}
p_values = []
for model in main_models:
    data["vulnerability_score"] += (
        df_least[n][model]["vulnerability_score"].tolist()
        + df_most[n][model]["vulnerability_score"].tolist()
    )
    data["Sycophancy"] += (["Least sycophancy"] * n) + (["Most sycophancy"] * n)
    data["Model"] += [model] * 2 * n
    p_values.append(
        ttest_ind(
            df_least[n][model]["vulnerability_score"].tolist(),
            df_most[n][model]["vulnerability_score"].tolist(),
        ).pvalue
    )
p_values = false_discovery_control(p_values)
idx = 0
for model in main_models:
    diff[model] = "*" if p_values[idx] < 0.05 else ""
    idx += 1

In [ ]:
fig, ax = plt.subplots()
ax = sns.barplot(
    pd.DataFrame(data),
    x="Model",
    y="vulnerability_score",
    hue="Sycophancy",
    hue_order=hue_order,
    order=model_order,
    palette=palette,
    saturation=1,
    ax=ax,
    capsize=0.2,
    err_kws={"linewidth": 1},
)
ax.set_ylabel("Vulnerability score")
ax.set_yticks(range(0, 6))
ax.set_xticklabels([size_fam[m][1] for m in model_order])
ax.set_xlabel("")
ax.tick_params(axis="x", length=0, pad=6)
style_axes(ax)

trans = ax.get_xaxis_transform()
start_x = 0
pad = 0.2
width = 0.8
for f, f_size in zip(fams, fam_size):
    x0 = start_x - (width / 2)
    x1 = start_x + ((f_size - 1) * (width + pad)) + (width / 2)
    start_x += ((f_size - 1) * (width + pad)) + width + pad
    ax.plot(
        [x0, x1],
        [-0.1, -0.1],
        color="#555555",
        lw=1.4,
        transform=trans,
        clip_on=False,
    )
    ax.text(
        (x0 + x1) / 2, -0.125, f, transform=trans, ha="center", va="top", fontsize=11.5
    )

heights = [4, 0, 0, 4.3, 4.1, 0]
for m in range(len(model_order)):
    if diff[model_order[m]]:
        ax.text(
            m, heights[m], "*", ha="center", va="center", fontsize=21, color="#333333"
        )

handles = [
    Patch(facecolor=palette["Least sycophancy"], label="Least sycophancy"),
    Patch(
        facecolor=palette["Most sycophancy"],
        label="Most sycophancy",
    ),
]
ax.legend(
    handles=handles,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.1),
    frameon=False,
    fontsize=11,
    columnspacing=1.2,
    handlelength=1.5,
    handletextpad=0.5,
    labelspacing=0.3,
)

plt.savefig(f"figures/vulnerability_score_20.pdf", bbox_inches="tight")
plt.show()

# Sycophancy per vulnerability score

## Load data

In [ ]:
models = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Qwen3-8B",
    "Qwen3.6-27B",
    "Qwen3.6-35B-A3B",
]
df_vulnerability = pd.read_csv(
    "data/personas_demographics_vulnerability_final (1).csv"
)[
    [
        "uuid",
        "inherent_vulnerability",
        "situational_vulnerability",
        "pathogenic_vulnerability",
        "vulnerability_score",
    ]
]
big_df = pd.DataFrame([])
for model in models:
    df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")

    original = df[df["prompt_type"] == "original_post"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
            "model_answer_constrained_no": "model_answer_constrained_no_original_post",
        }
    )
    print("got original")

    flipped = df[df["prompt_type"] == "flipped_story"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
            "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
        }
    )
    print("got flipped")

    df = original.merge(
        flipped,
        on=["persona_type", "persona_id", "prompt_id"],
        how="outer",
    )

    print("got result")
    df["sycophancy"] = (
        (
            df["model_answer_constrained_no_original_post"]
            > df["model_answer_constrained_yes_original_post"]
        )
        & (
            df["model_answer_constrained_no_flipped_story"]
            > df["model_answer_constrained_yes_flipped_story"]
        )
    ) * 100
    df = df.groupby(["persona_id", "persona_type"])["sycophancy"].mean().reset_index()
    df["model"] = model.capitalize().replace("-instruct", "").replace("-it", "")
    df = df.merge(df_vulnerability, left_on="persona_id", right_on="uuid")
    big_df = pd.concat([df, big_df], ignore_index=True)

## Correlation

In [ ]:
big_df[
    [
        "sycophancy",
        "inherent_vulnerability",
        "situational_vulnerability",
        "pathogenic_vulnerability",
        "vulnerability_score",
    ]
].corr()

## Figure

In [ ]:
def style_axes(ax):
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color="#E0E0E0", lw=1.4)
    ax.xaxis.grid(False)
    sns.despine(ax=ax)


model_order = [
    "Gemma-3-12b",
    "Gemma-3-27b",
    "Llama-3.1-8b",
    "Qwen3-8b",
    "Qwen3.6-27b",
    "Qwen3.6-35b-a3b",
]

In [ ]:
sns.set_theme(
    font_scale=2,
    style="ticks",
    rc={
        "font.family": "serif",
        "font.serif": ["DejaVu Serif"],
    },
)

g = sns.catplot(
    big_df.sort_values(by="vulnerability_score").rename(
        columns={
            "sycophancy": "Sycophancy",
            "model": "Model",
            "vulnerability_score": "Vulnerability Score",
        }
    ),
    kind="bar",
    col="Model",
    col_order=model_order,
    x="Vulnerability Score",
    y="Sycophancy",
    sharex=False,
    color="#E07B54",
    saturation=1,
    col_wrap=3,
    height=7,
    aspect=8 / 7,
)

for ax in g.axes.ravel():
    style_axes(ax)
    # add annotations
    for c in ax.containers:
        labels = [f"{(v.get_height()):.1f}" for v in c]
        ax.bar_label(c, labels=labels, label_type="edge", padding=1)
plt.savefig(f"figures/vulnerability_score_syco.pdf")
plt.show()